# Silver: Data Quality Validation

Validates silver layer data against configurable quality rules:

## Quality Checks

1. **Completeness**: Required fields must not be null
2. **Uniqueness**: Enterprise job IDs must be unique
3. **Referential integrity**: Valid source names
4. **Format validation**: URLs, dates, enums
5. **Business rules**: Salary ranges, location patterns

## Actions

- **PASS**: Record proceeds to next stage
- **WARN**: Log warning, proceed with flag
- **FAIL**: Route to quarantine table
- **REVIEW**: Queue for manual semantic review

In [0]:
dbutils.widgets.dropdown("validation_level", "standard", ["strict", "standard", "permissive"], "Validation Level")
dbutils.widgets.dropdown("quarantine_on_fail", "true", ["true", "false"], "Quarantine Failed Records")

validation_level = dbutils.widgets.get("validation_level")
quarantine_on_fail = dbutils.widgets.get("quarantine_on_fail") == "true"

print(f"Validation Level: {validation_level}")
print(f"Quarantine on Fail: {quarantine_on_fail}")

In [0]:
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"
METADATA_SCHEMA = f"{CATALOG}.metadata"
QUARANTINE_SCHEMA = f"{CATALOG}.quarantine"
QUARANTINE_SCHEMA = f"{CATALOG}.quarantine"

run_timestamp = F.current_timestamp()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

# Valid enum values
VALID_REMOTE_TYPES = ["REMOTE", "HYBRID", "ONSITE", "UNKNOWN"]
VALID_SOURCES = ["arbeitnow", "remotive"]  # Extend as sources are added

print(f"Run ID: {run_id}")

In [0]:
jobs_df = spark.table(f"{SILVER_SCHEMA}.silver_jobs_current") \
    .where("is_active = true AND soft_delete_flag = false")

record_count = jobs_df.count()
print(f"Records to validate: {record_count}")

if record_count == 0:
    dbutils.notebook.exit({"status": "success", "message": "No records to validate"})

In [0]:
from pyspark.sql.window import Window

def check_completeness(df):
    """Check required fields are not null"""
    required_fields = [
        "enterprise_job_id", 
        "source_name", 
        "source_job_id",
        "company_name_norm",
        "title_normalized",
        "record_hash"
    ]
    
    for field in required_fields:
        df = df.withColumn(
            f"dq_{field}_null",
            F.when(F.col(field).isNull(), F.lit("FAIL")).otherwise(F.lit("PASS"))
        )
    
    return df

def check_uniqueness(df):
    """Check enterprise_job_id is unique"""
    window = Window.partitionBy("enterprise_job_id")
    df = df.withColumn(
        "dq_unique_job_id",
        F.when(F.count("*").over(window) > 1, F.lit("FAIL")).otherwise(F.lit("PASS"))
    )
    return df

def check_referential_integrity(df):
    """Check source_name is valid"""
    df = df.withColumn(
        "dq_valid_source",
        F.when(F.col("source_name").isin(VALID_SOURCES), F.lit("PASS")).otherwise(F.lit("FAIL"))
    )
    return df

def check_format_validation(df):
    """Validate URL and enum formats"""
    # URL validation (simple check)
    df = df.withColumn(
        "dq_valid_url",
        F.when(
            F.col("source_url").isNull() | F.col("source_url").rlike("^https?://"),
            F.lit("PASS")
        ).otherwise(F.lit("WARN"))
    )
    
    # Remote type enum
    df = df.withColumn(
        "dq_valid_remote_type",
        F.when(
            F.col("remote_type").isin(VALID_REMOTE_TYPES),
            F.lit("PASS")
        ).otherwise(F.lit("FAIL"))
    )
    
    return df

def check_business_rules(df):
    """Business logic validation"""
    # Title length (should be reasonable)
    df = df.withColumn(
        "dq_title_length",
        F.when(
            (F.length(F.col("title_raw")) >= 3) & (F.length(F.col("title_raw")) <= 500),
            F.lit("PASS")
        ).otherwise(F.lit("WARN"))
    )
    
    # Company name not empty
    df = df.withColumn(
        "dq_company_not_empty",
        F.when(
            F.length(F.trim(F.col("company_name_raw"))) > 0,
            F.lit("PASS")
        ).otherwise(F.lit("FAIL"))
    )
    
    return df

print("Quality check functions loaded")

In [0]:
# Apply all quality checks
validated_df = jobs_df
validated_df = check_completeness(validated_df)
validated_df = check_uniqueness(validated_df)
validated_df = check_referential_integrity(validated_df)
validated_df = check_format_validation(validated_df)
validated_df = check_business_rules(validated_df)

# Collect all DQ columns
dq_columns = [c for c in validated_df.columns if c.startswith("dq_")]
print(f"Applied {len(dq_columns)} quality checks")

# Calculate overall status
fail_condition = F.array_contains(F.array(*[F.col(c) for c in dq_columns]), "FAIL")
warn_condition = F.array_contains(F.array(*[F.col(c) for c in dq_columns]), "WARN")

validated_df = validated_df.withColumn(
    "dq_overall_status",
    F.when(fail_condition, F.lit("FAIL"))
     .when(warn_condition, F.lit("WARN"))
     .otherwise(F.lit("PASS"))
)

# Add validation metadata
validated_df = validated_df.withColumn("dq_validated_at", run_timestamp)
validated_df = validated_df.withColumn("dq_validation_level", F.lit(validation_level))

print("Quality checks complete")

In [0]:
# Calculate summary metrics
metrics_df = validated_df.groupBy("dq_overall_status").count()

print("\n=== Data Quality Summary ===")
display(metrics_df)

# Detailed failure breakdown
if validated_df.where("dq_overall_status = 'FAIL'").count() > 0:
    print("\n=== Failure Details ===")
    failure_details = validated_df.where("dq_overall_status = 'FAIL'")
    
    for col in dq_columns:
        fail_count = failure_details.where(F.col(col) == "FAIL").count()
        if fail_count > 0:
            print(f"{col}: {fail_count} failures")

In [0]:
if quarantine_on_fail:
    # Quarantine failed records to workspace.quarantine.quarantine_jobs
    failed_records = validated_df.where("dq_overall_status = 'FAIL'")
    
    if failed_records.count() > 0:
        # Collect failed rule names for each record
        failed_rules = F.concat_ws(", ", F.array(*[
            F.when(F.col(c) == "FAIL", F.lit(c.replace("dq_", "")))
            for c in dq_columns
        ]))
        
        quarantine_df = failed_records.select(
            F.expr("uuid()").alias("quarantine_id"),
            F.col("source_name"),
            F.col("source_job_id"),
            F.col("enterprise_job_id"),
            F.lit("SILVER").alias("failure_stage"),
            failed_rules.alias("failed_rule_name"),
            F.lit("ERROR").alias("severity"),
            F.to_json(F.struct(*[c for c in jobs_df.columns])).alias("payload_json"),
            run_timestamp.alias("quarantined_at"),
            F.col("current_batch_id").alias("batch_id"),
            F.lit("PENDING").alias("reprocess_status"),
            F.lit(None).cast("string").alias("reprocess_batch_id"),
            F.lit(None).cast("timestamp").alias("resolved_at")
        )
        
        quarantine_df.write.format("delta").mode("append").saveAsTable(f"{QUARANTINE_SCHEMA}.quarantine_jobs")
        quarantined_count = quarantine_df.count()
        print(f"Quarantined {quarantined_count} failed records to {QUARANTINE_SCHEMA}.quarantine_jobs")
    else:
        quarantined_count = 0
else:
    quarantined_count = 0
    print("Quarantine disabled")

In [0]:
# Add DQ columns to current table (for tracking)
from delta.tables import DeltaTable

# Check if DQ columns exist, if not add them
try:
    current_schema = spark.table(f"{SILVER_SCHEMA}.silver_jobs_current").schema
    current_columns = [field.name for field in current_schema.fields]
    
    missing_columns = []
    if "dq_overall_status" not in current_columns:
        missing_columns.append("ADD COLUMN dq_overall_status STRING")
    if "dq_validated_at" not in current_columns:
        missing_columns.append("ADD COLUMN dq_validated_at TIMESTAMP")
    if "dq_validation_level" not in current_columns:
        missing_columns.append("ADD COLUMN dq_validation_level STRING")
    
    if missing_columns:
        for col_def in missing_columns:
            spark.sql(f"ALTER TABLE {SILVER_SCHEMA}.silver_jobs_current {col_def}")
        print(f"Added {len(missing_columns)} DQ tracking columns to current table")
    
    # Select only the DQ columns to update
    dq_update_df = validated_df.select(
        "enterprise_job_id",
        "dq_overall_status",
        "dq_validated_at",
        "dq_validation_level"
    )
    
    delta_current = DeltaTable.forName(spark, f"{SILVER_SCHEMA}.silver_jobs_current")
    
    # Merge DQ flags back
    delta_current.alias("cur").merge(
        dq_update_df.alias("dq"),
        "cur.enterprise_job_id = dq.enterprise_job_id"
    ).whenMatchedUpdate(
        set={
            "dq_overall_status": "dq.dq_overall_status",
            "dq_validated_at": "dq.dq_validated_at",
            "dq_validation_level": "dq.dq_validation_level"
        }
    ).execute()
    
    print("DQ flags updated in current table")
except Exception as e:
    print(f"Warning: Could not update DQ flags: {e}")

In [0]:
pass_count = validated_df.where("dq_overall_status = 'PASS'").count()
warn_count = validated_df.where("dq_overall_status = 'WARN'").count()
fail_count = validated_df.where("dq_overall_status = 'FAIL'").count()

result = {
    "status": "success",
    "run_id": run_id,
    "total_records": record_count,
    "pass_count": pass_count,
    "warn_count": warn_count,
    "fail_count": fail_count,
    "quarantined_count": quarantined_count,
    "pass_rate": round(pass_count / record_count * 100, 2) if record_count > 0 else 0
}

print("\n=== Validation Complete ===")
print(json.dumps(result, indent=2))

dbutils.notebook.exit(json.dumps(result))